# 📘 Capítulo 2 – Modelos Estatísticos Clássicos (ARIMA, SARIMA, SARIMAX, Holt-Winters)
**Autor:** Rodrigo Santana Ferreira  
**Disciplina:** Séries Temporais  

---
Este notebook contém os scripts Python apresentados no Capítulo 2, organizados por seção conforme o material da aula.

## 🔧 Instalação de Dependências

In [ ]:
# Instalação das bibliotecas necessárias
!pip install pandas matplotlib statsmodels pmdarima

## 📦 Passo 1 – Preparação do Ambiente e Carregamento de Dados
Carregamento da série AirPassengers e visualização inicial (seção HANDS ON).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

# Exemplo 1: Série clássica AirPassengers (carregue ou use statsmodels)
# url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
# df = pd.read_csv(url, parse_dates=['Month'], index_col='Month')
# series = df['Passengers']

# Para teste rápido, use um dataset embutido ou simule
from statsmodels.datasets import get_rdataset
air = get_rdataset("AirPassengers").data
air.index = pd.date_range(start='1949-01', periods=len(air), freq='M')
series = air['value'].astype(float)

plt.figure(figsize=(12, 5))
plt.plot(series)
plt.title('Série Original: AirPassengers')
plt.show()

## 🔍 Passo 2 – Verificação de Estacionariedade (Visual + Teste ADF)
Aplicação de log + diferenciação para tornar a série estacionária.

In [ ]:
# Teste ADF
adf_result = adfuller(series)
print(f'ADF Statistic: {adf_result[0]}')
print(f'p-value: {adf_result[1]}')  # p > 0.05 → não estacionária

# Aplicar log + 1 diferença não-sazonal + 1 sazonal (para mensal)
log_series = np.log(series)
diff1 = log_series.diff().dropna()
diff_seasonal = diff1.diff(12).dropna()  # D=1, s=12

# Plote a série transformada
plt.figure(figsize=(12, 5))
plt.plot(diff_seasonal)
plt.title('Série após log + Δ¹ + Δ¹₂ (estacionária)')
plt.show()

## 📈 Passo 3 – Plotar ACF e PACF da Série Estacionária
Identificação dos parâmetros p, q, P, Q pelos gráficos de autocorrelação.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# ACF
plot_acf(diff_seasonal, lags=40, ax=axes[0], title='ACF da série diferenciada')
axes[0].set_xlabel('Lag')
axes[0].set_ylabel('Correlação')

# PACF
plot_pacf(diff_seasonal, lags=40, ax=axes[1], title='PACF da série diferenciada', method='ywm')
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('Correlação Parcial')

plt.tight_layout()
plt.show()

## 🛠️ Passo 5 – Ajuste Inicial do Modelo SARIMA
Exemplo com o modelo airline clássico: SARIMA(0,1,1)(0,1,1)₁₂.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Exemplo candidato: SARIMA(0,1,1)(0,1,1)12 – modelo airline clássico
model = SARIMAX(log_series, order=(0,1,1), seasonal_order=(0,1,1,12))
results = model.fit()
print(results.summary())

# Diagnóstico: resíduos devem ser ruído branco
plot_acf(results.resid, lags=40, title='ACF dos Resíduos')
plt.show()

## ⚙️ Passo 6 – Automatizar com pmdarima (auto_arima)
Seleção automática de parâmetros SARIMA recomendada para produção.

In [ ]:
from pmdarima import auto_arima

auto_model = auto_arima(log_series, seasonal=True, m=12, trace=True)
print(auto_model.summary())

## 🔬 Diagnóstico de Resíduos – Passo a Passo
Verificação formal de que os resíduos se comportam como ruído branco (seção SAIBA MAIS).

In [ ]:
# Passo 1: Ajuste um modelo candidato (exemplo SARIMA clássico)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.datasets import get_rdataset

# Carregar AirPassengers (exemplo clássico)
air = get_rdataset("AirPassengers").data
air.index = pd.date_range(start='1949-01', periods=len(air), freq='M')
series = air['value'].astype(float)
log_series = np.log(series)  # transformação para estabilizar variância

# Ajuste SARIMA(0,1,1)(0,1,1)12 – modelo airline
model = SARIMAX(log_series, order=(0,1,1), seasonal_order=(0,1,1,12))
results = model.fit(disp=False)
print(results.summary())

In [ ]:
# Passo 2: Extrair e plotar resíduos
residuals = results.resid

plt.figure(figsize=(12, 8))

# Resíduos ao longo do tempo
plt.subplot(311)
plt.plot(residuals)
plt.title('Resíduos do modelo')
plt.axhline(0, color='red', linestyle='--')

# Histograma + densidade (verificar normalidade)
plt.subplot(312)
plt.hist(residuals, bins=20, density=True, alpha=0.6, color='g')
residuals.plot(kind='kde', color='blue')
plt.title('Histograma e densidade dos resíduos')

# QQ-plot (comparar com normal teórica)
plt.subplot(313)
from statsmodels.graphics.gofplots import qqplot
qqplot(residuals, line='s', ax=plt.gca())
plt.title('QQ-plot dos resíduos')

plt.tight_layout()
plt.show()

In [ ]:
# Passo 3: Testes formais de diagnóstico

# 1. Teste Ljung-Box: autocorrelação serial nos resíduos (H0: sem autocorrelação)
from statsmodels.stats.diagnostic import acorr_ljungbox
lb_test = acorr_ljungbox(residuals, lags=[10, 20], return_df=True)
print("Ljung-Box test:\n", lb_test)
# p-value > 0.05 em vários lags → resíduos sem correlação serial

# 2. Teste Jarque-Bera: normalidade dos resíduos
from statsmodels.stats.stattools import jarque_bera
jb_test = jarque_bera(residuals)
print(f"Jarque-Bera: statistic={jb_test[0]:.2f}, p-value={jb_test[1]:.4f}")
# p-value > 0.05 → resíduos aproximadamente normais

# 3. ACF dos resíduos (deve estar dentro das bandas de confiança)
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(residuals, lags=40, title='ACF dos Resíduos')
plt.show()

## 📊 Comparação de Modelos via AIC / BIC
Seleção do melhor modelo SARIMA com base em critérios de informação (seção SAIBA MAIS).

In [ ]:
# Candidatos comuns para AirPassengers (após log)
candidates = [
    (0,1,1, 0,1,1, 12),   # clássico airline
    (1,1,1, 0,1,1, 12),
    (0,1,2, 0,1,1, 12),
    (2,1,2, 1,1,1, 12),
]

results_list = []
for order, s_order in [(o[:3], o[3:]) for o in candidates]:
    try:
        mod = SARIMAX(log_series, order=order, seasonal_order=s_order)
        res = mod.fit(disp=False)
        print(f"Order {order}{s_order} | AIC: {res.aic:.2f} | BIC: {res.bic:.2f}")
        results_list.append((order, s_order, res.aic, res.bic, res))
    except:
        print(f"Order {order}{s_order} falhou")

In [ ]:
# Passo 2: Escolher o melhor

# Ordenar por AIC (ou BIC – BIC penaliza mais)
best_aic = min(results_list, key=lambda x: x[2])
best_bic = min(results_list, key=lambda x: x[3])
print(f"Melhor por AIC: Order {best_aic[0]}{best_aic[1]} | AIC = {best_aic[2]:.2f}")
print(f"Melhor por BIC: Order {best_bic[0]}{best_bic[1]} | BIC = {best_bic[3]:.2f}")

In [ ]:
# Alternativa automática (recomendada para produção)
from pmdarima import auto_arima

auto_model = auto_arima(log_series, seasonal=True, m=12, trace=True,
                        error_action='ignore', suppress_warnings=True)
print(auto_model.summary())  # Mostra AIC/BIC do melhor encontrado

## 🌡️ Tutorial: Ajuste de Holt-Winters
Suavização exponencial tripla — aditiva e multiplicativa (seção SAIBA MAIS).

In [ ]:
# Passo 1: Ajuste aditivo e multiplicativo
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Holt-Winters Aditivo (tendência + sazonalidade constante)
model_add = ExponentialSmoothing(series,  # série original (sem log aqui)
                                  trend='add',
                                  seasonal='add',
                                  seasonal_periods=12).fit()

# Holt-Winters Multiplicativo (sazonalidade proporcional ao nível)
model_mul = ExponentialSmoothing(series,
                                  trend='add',
                                  seasonal='mul',
                                  seasonal_periods=12).fit()

print("Aditivo - AIC:", model_add.aic)
print("Multiplicativo - AIC:", model_mul.aic)

In [ ]:
# Passo 2: Previsão e comparação visual

# Previsão de 24 meses à frente
forecast_add = model_add.forecast(24)
forecast_mul = model_mul.forecast(24)

plt.figure(figsize=(12, 6))
plt.plot(series, label='Observado')
plt.plot(forecast_add, label='Holt-Winters Aditivo')
plt.plot(forecast_mul, label='Holt-Winters Multiplicativo')
plt.title('Previsão Holt-Winters vs Observado')
plt.legend()
plt.show()